# fractional-stride-zero-insertion — ex2: predict stride-S ConvT output size from the shape formula

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `fractional-stride-zero-insertion`. Running the final beacon cell reports progress against the `CNN: ConvT fractional-stride zero insertion` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT fractional-stride zero insertion` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`fractional-stride-zero-insertion`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "fractional-stride-zero-insertion"
DD_SUBTOPIC = "CNN: ConvT fractional-stride zero insertion"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Stride-S ConvT output shape (no padding) — quick refresher

With `padding=0` and `output_padding=0`, a `nn.ConvTranspose2d` with kernel `K` and stride `S` maps a 1-D input of length `H_in` to:

```
H_out = (H_in - 1) * S + K
```

This is the **adjoint** of forward `Conv2d(stride=S, K)` (which goes `H_in -> (H_in - K) // S + 1`). The fractional-stride view: dilate the input by inserting `S - 1` zeros between every pair of input pixels — intermediate length `(H_in - 1) * S + 1` — then apply a stride-1 K-tap conv, adding `K - 1`.

**Exemplar.** `H_in = 4, K = 3, S = 2 -> (4-1)*2 + 3 = 9`. Plug into `nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2)(t.randn(1,1,4,4))` and the spatial axis is 9.

### Exercise 2 — predict stride-S ConvT output size from the shape formula

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the no-padding ConvT shape formula `H_out = (H_in - 1) * S + K` to predict the spatial output size of `nn.ConvTranspose2d` for given `(H_in, K, S)` and verify against the real module.
> Keywords: ConvTranspose2d, fractional-stride, output-shape, no-padding
> ```

**KCs targeted:** `convT-stride-shape-formula`, `convT-stride-zero-dilation`

Implement `ex2_convT_no_pad_outlen(h_in, k, s)`. Return the spatial output length of a `nn.ConvTranspose2d` with kernel size `k`, stride `s`, no padding, no output_padding, applied to a 1-D input of length `h_in`.

**Formula.**
```
h_out = (h_in - 1) * s + k
```

**Why.** The stride-S ConvT first DILATES the input by inserting `s - 1` zeros between every pair of adjacent input pixels — giving an intermediate length `(h_in - 1) * s + 1`. Then it applies a stride-1 K-tap conv with full `(K - 1)` implicit padding on each side, which adds `K - 1` more. The two effects compose as `(h_in - 1) * s + 1 + (K - 1) = (h_in - 1) * s + K`.

**Adjoint relationship.** Forward `Conv2d(stride=s, K)` maps `H -> (H - K)//s + 1` (shrinks). Its adjoint ConvT goes the other way at the same stride.

The test exercises many `(h_in, k, s)` combinations and compares your prediction against the actual output shape of `nn.ConvTranspose2d` (stride-1, 2, 3 cases).

In [ ]:
def ex2_convT_no_pad_outlen(h_in: int, k: int, s: int) -> int:
    """Output length of nn.ConvTranspose2d with no padding."""
    raise NotImplementedError()


def _test_ex2():
    from torch import nn

    # Direct value checks.
    assert ex2_convT_no_pad_outlen(h_in=4, k=3, s=1) == 6,  '(4-1)*1 + 3 = 6'
    assert ex2_convT_no_pad_outlen(h_in=4, k=3, s=2) == 9,  '(4-1)*2 + 3 = 9'
    assert ex2_convT_no_pad_outlen(h_in=5, k=3, s=2) == 11, '(5-1)*2 + 3 = 11'
    assert ex2_convT_no_pad_outlen(h_in=8, k=4, s=2) == 18, '(8-1)*2 + 4 = 18'
    assert ex2_convT_no_pad_outlen(h_in=1, k=5, s=1) == 5,  'single pixel + 5-tap → 5'
    assert ex2_convT_no_pad_outlen(h_in=1, k=3, s=4) == 3,  'single-pixel input: stride has no effect'
    assert ex2_convT_no_pad_outlen(h_in=3, k=2, s=3) == 8,  '(3-1)*3 + 2 = 8'

    # Monotonicity sanity — output strictly increases with each input dim.
    assert ex2_convT_no_pad_outlen(5, 3, 2) > ex2_convT_no_pad_outlen(4, 3, 2), 'larger h_in → larger output'
    assert ex2_convT_no_pad_outlen(5, 5, 2) > ex2_convT_no_pad_outlen(5, 3, 2), 'larger k → larger output'
    assert ex2_convT_no_pad_outlen(5, 3, 3) > ex2_convT_no_pad_outlen(5, 3, 2), 'larger s → larger output'

    # Stride-S adjoint round-trip sanity: forward Conv2d with stride S and K, no pad:
    #   H_in → (H_in - K) // S + 1
    # Plug back into ConvT with the SAME S, K, no pad: lands at >= H_in for any divisible H_in.
    # We just check that no-padding ConvT yields the canonical 'dilate + stride-1 conv' shape.
    cases = [(4, 3, 1), (4, 3, 2), (5, 3, 2), (8, 4, 2), (16, 3, 1), (10, 5, 2), (7, 3, 2), (1, 3, 4), (3, 2, 3)]
    for h_in, k, s in cases:
        ct = nn.ConvTranspose2d(in_channels=1, out_channels=1, kernel_size=k, stride=s, padding=0)
        x = t.randn(1, 1, h_in, h_in)
        actual = ct(x).shape[-1]
        predicted = ex2_convT_no_pad_outlen(h_in, k, s)
        assert predicted == actual, (
            f'(h={h_in},k={k},s={s}): predicted {predicted}, actual {actual}'
        )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_convT_no_pad_outlen(h_in: int, k: int, s: int) -> int:
    return (h_in - 1) * s + k
```

**No-padding form is the cleanest entry into the full formula.** The full PyTorch shape rule (with `padding=P`, `output_padding=OP`, `dilation=D`) is `h_out = (h_in - 1) * s - 2 * p + d * (k - 1) + op + 1`. With `D = 1`, `P = 0`, `OP = 0`, this collapses to `(h_in - 1) * s + k`. Master this base case, then layer on padding (`- 2 * p`) and output_padding (`+ op`) as separate corrections.

**Why ConvT(K=4, S=2, P=1, OP=0) does NOT clean-2× upsample.** Plug in: `(h - 1) * 2 - 2 + 4 = 2 * h`. That's clean 2× — but ONLY when `P=1`. Drop the padding and you get `(h - 1) * 2 + 4 = 2h + 2` — two extra pixels. The padding argument is what makes the upsample clean; this drill is the no-padding baseline you correct from.

**Composes with `padding-amount-formula-convT` ex1.** That drill adds the `- 2 * p` correction. The combined formula covers every shape question a stride-S decoder block can ask.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()